<a href="https://colab.research.google.com/github/mkrauter/TrussGame/blob/master/truss_game_v3_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Truss game v3: teaching a network the mechanics

A random truss is pinned at its two outermost nodes and loaded at one of the
others. Where does the loaded node end up?

This notebook trains the model that plays that game at **96%**, and — more
usefully — shows why the obvious architecture only reaches 77% and gets *worse*
when you apply the textbook fix to it.

**What you should get out of it**

1. Why a convolutional network is the wrong shape for this problem, with the
   measurement that says so.
2. Why message passing is the right shape: solving `K u = f` iteratively *is*
   message passing on the graph of the structure.
3. How to write a loss that needs no labels, and how to test it before you
   trust it.
4. How to turn "difficulty" into something physically meaningful rather than
   injected noise.

**Runtime:** about ten minutes end to end on a free Colab GPU. The corpus is
20 MB and takes a second and a half to build; the model is 42,498 parameters.

**A note on how this notebook is built.** It clones the repository and imports
the real `trussnet` package — no model code is pasted into these cells. The
predecessor notebook in this repo did paste its code, drifted from what actually
shipped, and now reports training-set accuracy for a model it cannot reproduce.
That is the failure mode this structure avoids.

In [ ]:
# Works both in Colab (clones the repo) and locally (uses the checkout you are in).
import os, subprocess, sys
from pathlib import Path

def repo_root():
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / 'training' / 'trussnet').is_dir():
            return candidate
    return None

ROOT = repo_root()
if ROOT is None:
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/mkrauter/TrussGame.git'], check=True)
    ROOT = Path.cwd() / 'TrussGame'

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'training'))
print('repository:', ROOT)

# The corpus generator is plain JavaScript -- it runs the same truss code the
# game runs, so the physics here cannot drift from the physics you play against.
if subprocess.run(['node', '--version'], capture_output=True).returncode != 0:
    print('installing node...')
    subprocess.run('apt-get -qq update && apt-get -qq install -y nodejs', shell=True, check=True)
print('node:', subprocess.run(['node', '--version'], capture_output=True, text=True).stdout.strip())

# Colab ships both of these; a local Jupyter might not.
for package in ('torch', 'matplotlib'):
    try:
        __import__(package)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], check=True)

import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())

## 1. The task is smaller than it looks

Every truss in this game is **10 nodes**, about 21 members, and **16 free
degrees of freedom**. The whole problem is determined by **21 numbers**: 20 node
coordinates plus which node is loaded. The supports are the extreme-x nodes.
The connectivity is the Delaunay triangulation of those coordinates. The load is
a constant, pointing down.

Given that, the answer is a 16x16 linear solve:

$$K u = f$$

where `K` is assembled by summing one 4x4 block per member. Nothing about this is
statistical — it is exact. What we are asking a network to learn is the *solution
operator*, `K⁻¹`.

In [ ]:
# Build the corpus: 20,000 trusses for training, 2,000 for validation, from
# disjoint seed ranges. This runs the game's own solver.
!node training/generate_graph_corpus.mjs --split train --count 20000 --seed-base 0
!node training/generate_graph_corpus.mjs --split val   --count 2000  --seed-base 1000000

In [ ]:
import numpy as np
from trussnet import graph_data, metrics

meta, samples = graph_data.load_raw('val')
print(f"{len(samples)} validation trusses")

nodes = np.array([s['nodes'] for s in samples])
elements = [s['elements'] for s in samples]
print(f"nodes per truss:    {nodes.shape[1]}")
print(f"members per truss:  {np.mean([len(e) for e in elements]):.1f} on average")
print(f"free DOF:           {(nodes.shape[1] - 2) * 2}")

In [ ]:
import matplotlib.pyplot as plt

def draw(sample, ax):
    """Undeformed structure in grey, settled position in colour."""
    pts = np.array(sample['nodes'])
    moved = pts + np.array(sample['displacement'])
    for a, b in sample['elements']:
        ax.plot(*zip(pts[a], pts[b]), color='0.75', lw=1, zorder=1)
        ax.plot(*zip(moved[a], moved[b]), color='0.25', lw=1.4, zorder=2)
    ax.scatter(*pts[sample['supports']].T, marker='^', s=90, c='tab:green', zorder=3)
    n = sample['loadedNode']
    ax.annotate('', xy=moved[n], xytext=pts[n],
                arrowprops=dict(arrowstyle='->', color='tab:red', lw=1.6), zorder=4)
    ax.scatter(*pts[n], marker='o', s=60, c='tab:blue', zorder=5)
    ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off')

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, s in zip(axes, samples[:3]):
    draw(s, ax)
fig.suptitle('grey = as drawn,  dark = where it settles,  red arrow = what you must predict')
plt.show()

## 2. Baselines first

The scoring rule is relative: **100%** means you clicked exactly where the node
settled, **0%** means you missed by at least as far as it travelled. So the score
is the fraction of the movement you captured — which means *guessing the starting
point scores exactly zero*, however small the movement was.

That property makes naive baselines essential. A model can look respectable and
still be worthless, and you cannot tell without the comparison.

In [ ]:
train = graph_data.load_targets('train')
val = graph_data.load_targets('val')

fitted = metrics.fit_baselines(train)          # constants fitted on train only
scored = {name: metrics.accuracy(val['start'], val['end'], guess).mean()
          for name, guess in metrics.baseline_predictions(val, fitted).items()}

fair = {k: v for k, v in scored.items() if not k.startswith('[oracle]')}
print('baselines -- nothing here is told any part of the answer:')
for name, score in sorted(fair.items(), key=lambda kv: -kv[1]):
    print(f'  {score:5.1f}%   {name}')

print()
print('oracles -- these ARE told part of the answer, so they bound what is available:')
for name, score in sorted(scored.items(), key=lambda kv: -kv[1]):
    if name.startswith('[oracle]'):
        print(f'  {score:5.1f}%   {name.replace("[oracle] ", "")}')

Roughly **59.5%** for "drop it straight down by the average distance, scaled by
the support span". That is the number to beat, and it is higher than people
expect — most of this game's score is available without understanding any
mechanics at all.

One more number worth knowing: an oracle that predicts *straight down* but with
the **true** magnitude scores **79%**. So getting the distance right while never
reasoning about direction already reaches the top of the human band. Magnitude is
where the information is.

## 3. The negative result: a CNN on screenshots

The obvious approach is to hand a convolutional network a picture of the truss
and regress the answer. That was v1 and v2 of this project, and a careful
re-implementation of it reaches **77%**.

Its failure is specific and instructive. The support span is 190–233px at 256px
input, and the network's receptive field was 106px — no unit in it could ever see
both supports at once. That looks like a textbook diagnosis with a textbook fix:
widen the receptive field. So it was widened to 405px, using dilation, at an
*identical* parameter count.

| architecture | receptive field | score |
|---|---|---|
| 5 stages, dilation 1,1,1,2,4 | 237px | 77.9% |
| 5 stages, dilation 1,1,2,4,8 | 405px | **77.2%** |

It got **worse**. Not dramatically, but reliably — and the version that started
worst was the one that could see furthest.

**Why.** `K` is sparse: one block per member. But its inverse is *dense*. Every
node's settled position depends on every other node in the structure, because
that is what solving a linear system means. A stack of local operators, however
wide you make its field, is the wrong computational class for a global implicit
solve. Receptive field was never the binding constraint, so buying more of it
bought nothing.

*(These numbers come from `training/train.py`, which needs the rendered pixel
corpus. Reproducing them here would mean a headless browser and 500 MB of PNGs,
so they are quoted rather than re-run.)*

## 4. Message passing is the right shape

Here is the observation the architecture is built on.

Iterative solvers for `K u = f` — Jacobi, Gauss–Seidel, conjugate gradient — all
work by repeatedly exchanging information between variables that share an
equation. For a truss, two nodes share an equation exactly when a member joins
them. **So an iterative solve is message passing on the graph of the structure**,
and one round of message passing is one iteration.

That gives a concrete design:

- **Nodes** carry their position, their role (free / support / loaded), and the
  applied force.
- **Edges** carry the exact entries of the element stiffness matrix,
  $\frac{1}{\hat{L}}\begin{bmatrix} c^2 & cs \\ cs & s^2 \end{bmatrix}$.
  The network is handed the ingredients of `K` and only has to learn to invert
  it — it never has to rediscover the stiffness formula from geometry.
- **The processor** runs T rounds with **weights shared across rounds**, so a
  round is genuinely an iteration rather than just another layer.
- **Supports are pinned exactly**, not learned approximately. A boundary
  condition is knowledge, not something to fit.
- Everything is **non-dimensionalised by the support span**, which removes the
  scale degree of freedom exactly: scaling a truss scales its displacements by
  the same factor, so the network never has to learn that.

In [ ]:
from trussnet.gnn import TrussGNN, count_parameters, physics_residual

model = TrussGNN(hidden=64, rounds=10)
print(f'{count_parameters(model):,} parameters, {model.rounds} message-passing rounds')

# The graph diameter is 3, so information crosses any truss in 3 rounds.
# More rounds buy accuracy, not reach -- exactly as with a solver.
ds = graph_data.TrussGraphs('val', sigma=0.0)
item = ds[0]
print('\nnode features :', item['node_feat'].shape, '(free, support, loaded, fx, fy, x, y)')
print('edge features :', item['edge_feat'].shape, '(c^2, cs, s^2, 1/L)')
print('\nfirst three edges, i.e. literal entries of K:')
print(item['edge_feat'][:3])

## 5. A loss that needs no labels

We know the equation the answer must satisfy. So besides the supervised loss, we
can score a prediction on whether it is in **equilibrium**:

$$r_i = \sum_{j} \frac{1}{\hat{L}_{ij}} M_{ij} (\hat{u}_i - \hat{u}_j) - \hat{f}_i$$

This needs no ground truth at all — it is computed from the inputs. It is also
the kind of code that is easy to get subtly wrong, so **test it against an answer
you already know** before training anything on it.

In [ ]:
import torch
from torch.utils.data import DataLoader

batch = next(iter(DataLoader(ds, batch_size=64)))
exact = batch['target']                      # the true displacement field

print(f"exact field          -> max |Ku - f| = {physics_residual(exact, batch).abs().max():.2e}")
print(f"field scaled by 0.9  -> max |Ku - f| = {physics_residual(exact * 0.9, batch).abs().max():.3f}")
print(f"zero field           -> max |Ku - f| = {physics_residual(exact * 0, batch).abs().max():.3f}")

Zero on the exact answer, and proportional to the error otherwise — so the term
means what we think it means.

Worth dwelling on: the first version of this function *added* the applied load
where it should have subtracted it. Trained on, it would have quietly pulled
every prediction toward a wrong equilibrium. The test above catches it
immediately, because the wrong sign returns exactly `2f` rather than zero. A loss
you have not evaluated at a known answer is a loss you are trusting on faith.

## 6. Train it

Two supervision signals, plus the equilibrium term:

- the **full displacement field** — all 8 free nodes, not just the loaded one.
  The solver computes them anyway, and throwing them away costs you 8x the
  signal from the same trusses.
- **perceptual noise** on the coordinates, and an occasional wrong member. The
  deployed model reads the structure off a screenshot, so it should train on the
  kind of input it will actually receive rather than on perfect graphs.

We run the real training script rather than re-implementing its loop here.

In [ ]:
# sys.executable, not bare `python`: the kernel's interpreter is the one with torch.
!{sys.executable} training/train_gnn.py --epochs 25 --sigma 0.9 --member-noise 0.12

In [ ]:
import json
from pathlib import Path

run = sorted(Path('training/runs_gnn').glob('*/history.json'),
             key=lambda p: p.stat().st_mtime)[-1]
history = json.loads(run.read_text())

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
epochs = [h['epoch'] for h in history]
left.plot(epochs, [h['mean'] for h in history], marker='o', ms=3)
left.axhline(59.5, ls='--', c='0.6')
left.text(epochs[-1], 60.5, 'straight-down baseline', ha='right', c='0.4')
left.set_xlabel('epoch'); left.set_ylabel('validation score (%)'); left.set_title('accuracy')
right.plot(epochs, [h['residual'] for h in history], marker='o', ms=3, c='tab:red')
right.set_xlabel('epoch'); right.set_ylabel('mean |Ku - f|')
right.set_title('equilibrium residual (fraction of applied load)')
for ax in (left, right):
    ax.grid(alpha=0.3)
plt.show()

print(f"best validation score: {max(h['mean'] for h in history):.1f}%")

Notice that the two curves move together. The network is not just fitting the
targets — it is converging on solutions that satisfy the governing equation. That
is the sign that it learned the mechanics rather than a correlation.

## 7. Difficulty is solver iterations

Because the rounds share weights, we can run **fewer at inference than we trained
with**. That does not damage the model; it gives a less-converged solver, which
is a physically meaningful notion of a weaker opponent.

In [ ]:
from trussnet.gnn import TrussGNN

device = 'cuda' if torch.cuda.is_available() else 'cpu'
checkpoint = torch.load(run.parent / 'best.pt', map_location=device, weights_only=False)
trained = TrussGNN(hidden=checkpoint['config']['hidden'],
                   rounds=checkpoint['config']['rounds']).to(device)
trained.load_state_dict(checkpoint['model']); trained.eval()

loader = DataLoader(graph_data.TrussGraphs('val', sigma=checkpoint['config']['sigma'], seed=1),
                    batch_size=256)
targets = graph_data.load_targets('val')

def score_at(rounds):
    clicks, residuals = [], []
    with torch.no_grad():
        for b in loader:
            b = {k: v.to(device) for k, v in b.items()}
            pred = trained(b, rounds=rounds)
            residuals.append(physics_residual(pred, b).abs().amax(dim=(1, 2)).cpu())
            idx = b['loaded'].view(-1, 1, 1).expand(-1, 1, 2)
            u = pred.gather(1, idx).squeeze(1) * b['span'].unsqueeze(-1)
            clicks.append((b['seen_start'] + u).cpu())
    clicks = torch.cat(clicks).numpy()
    scores = metrics.accuracy(targets['start'], targets['end'], clicks)
    return scores.mean(), float(torch.cat(residuals).mean())

sweep = [(r, *score_at(r)) for r in (1, 2, 3, 4, 6, 8, 10)]
for r, s, res in sweep:
    print(f'{r:>3} rounds   {s:5.1f}%   |Ku-f| {res:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([r for r, _, _ in sweep], [s for _, s, _ in sweep], marker='o')
ax.axhline(59.5, ls='--', c='0.6'); ax.text(1, 61, 'baseline', c='0.4')
ax.axhspan(70, 80, color='tab:green', alpha=0.12)
ax.text(1, 74, 'human band', c='tab:green')
ax.set_xlabel('message-passing rounds at inference')
ax.set_ylabel('score (%)'); ax.grid(alpha=0.3)
ax.set_title('one dial, from novice to expert')
plt.show()

The residual falls in step with the score, which is what tells you this is a
solver converging and not a model degrading. In the game, `[` and `]` move along
this curve.

## 8. The other half: perception

The deployed opponent never receives this graph. Its only input is the colour
screenshot a player looks at, and it recovers the structure from that before any
of the above happens — a small keypoint network finds the nodes and their roles,
and connectivity is read by checking which node pairs have a line drawn between
them.

Splitting it that way matters, and it is worth being precise about *why* it is
fair rather than hand-waving. A perfect solver reading node positions with a
given error scores:

| perceived node error | score |
|---|---|
| 0px (exact) | 100% |
| 1px | 99.0% |
| 3px | 97.1% |
| 12px | 88.2% |

A human reads a marker on a 900px canvas to about 1–3px. So **exact geometry is
worth roughly 2 points over human-level perception, while doing the mechanics at
all is worth 40**. Perception is not where this game is won, which is exactly why
it is worth separating it out and giving each half the architecture that suits it.

The full pipeline scores **96.3%** end to end from pixels. Its remaining error is
almost entirely perceptual: a flawless solver with the same eyes would score
about 98%.

You can play it at
[mkrauter.github.io/TrussGame](https://mkrauter.github.io/TrussGame/) — press
**V** in v3 to see the structure the AI recovered from the screen.

## What to take away

1. **Measure a naive baseline before you believe any model.** 59.5% here comes
   from one line of arithmetic, and both earlier models spent years not clearly
   beating it.
2. **Match the architecture to the computational class of the problem.** Not to
   its input format. The picture was solvable; convolutions were not the thing
   that could solve it, and widening their receptive field — the obvious fix —
   made it worse.
3. **Give the network what you already know.** Exact stiffness entries as edge
   features, boundary conditions imposed rather than learned, scale removed by
   non-dimensionalisation. Every one of those is capacity not spent rediscovering
   something you could have written down.
4. **Test a loss at a known answer before training on it.**
5. **Supervise everything you computed.** The full displacement field was sitting
   there unused, worth 8x the signal for free.